# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup, population and scoring\n\n*Clones my repo on Colab, rebuilds the Week-5/6 data and label, and scores every page with a model that never saw its client.*

In [1]:
# NOTE: Run all takes about 1-3 minutes. Nothing in here is slow, but please don't interrupt it.
import os, sys, subprocess, json, warnings
import numpy as np
import pandas as pd
import sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
os.makedirs("work/outputs", exist_ok=True)     # lowercase 'outputs' on purpose
os.makedirs("work/figures", exist_ok=True)

SEED = 42
np.random.seed(SEED)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape, "| scikit-learn", sklearn.__version__, "| seed", SEED)

Rows, columns: (30000, 44) | scikit-learn 1.6.1 | seed 42


In [2]:
# ---- Windows (same idea as Weeks 5-6) ----
#   OUTCOME window  = last 30 days
#   FEATURE window  = days 31-90 back  (= 90-day total minus last 30 days)
#       inside it:  "prev 30" = days 31-60 back,  "early" = days 61-90 back
df["pre_impr"]   = df["impressions_90d"] - df["impressions_last_30d"]
df["pre_clk"]    = df["clicks_90d"] - df["clicks_last_30d"]
df["early_impr"] = df["pre_impr"] - df["impressions_prev_30d"]
assert (df[["pre_impr", "pre_clk", "early_impr"]] >= 0).all().all()

# ---- The queue population is chosen with information available BEFORE the outcome window ----
# (Week 6 flagged that requiring 500 impressions in the outcome window used future information.)
P = df[(df["pre_impr"] >= 500) & (df["avg_position"] > 0)].reset_index(drop=True)
P["labelled"] = P["impressions_last_30d"] >= 500        # used ONLY to decide who can be scored for validation
print(f"Queue population (visible enough before the outcome window): {len(P):,} pages, {P['client_id'].nunique()} clients")
print(f"  of which {P['labelled'].sum():,} also have >= 500 impressions in the outcome window (the Week-5/6 validated population)")

# ---- Label (same definition as Week 5) and tier yardsticks ----
L = P[P["labelled"]]
bench = L.groupby("position_tier").apply(lambda x: x["clicks_last_30d"].sum() / x["impressions_last_30d"].sum())
P["expected_last30"] = P["position_tier"].map(bench) * P["impressions_last_30d"]
P["y"] = np.where(P["labelled"], (P["clicks_last_30d"] <= 0.5 * P["expected_last30"]).astype(float), np.nan)
P["realized_missed_30d"] = (P["expected_last30"] - P["clicks_last_30d"]).clip(lower=0)      # outcome window, evaluation only

# Feature-window yardstick: pooled CTR per tier over the WHOLE queue population, feature window only
pre_rate = P.groupby("position_tier").apply(lambda x: x["pre_clk"].sum() / x["pre_impr"].sum())
P["pre_ctr_pct"] = P["pre_clk"] / P["pre_impr"] * 100
P["rule_score"]  = (P["position_tier"].map(pre_rate) * P["pre_impr"] - P["pre_clk"]).clip(lower=0)
FEATURE_DAYS = 60
P["est_missed_clicks_30d"] = P["rule_score"] * 30 / FEATURE_DAYS     # clicks "at stake" per 30 days, from the feature window only

# ---- Features: identical to my Week-5 / Week-6 honest set ----
P["log_pre_impr"] = np.log1p(P["pre_impr"]); P["log_pre_clk"] = np.log1p(P["pre_clk"])
P["log_prev30_impr"] = np.log1p(P["impressions_prev_30d"])
P["prev30_ctr_pct"] = np.where(P["impressions_prev_30d"] > 0, P["clicks_prev_30d"] / P["impressions_prev_30d"].clip(lower=1) * 100, 0)
P["has_word_count"] = P["word_count"].notna().astype(int); P["word_count_f"] = P["word_count"].fillna(P["word_count"].median())
P["has_volume"] = P["search_volume"].notna().astype(int); P["log_volume"] = np.log1p(P["search_volume"].fillna(0))
P["competition_f"] = P["competition"].fillna(0); P["cpc_f"] = P["cpc"].fillna(0); P["log_age"] = np.log1p(P["content_age_days"])
STATIC = ["avg_position", "log_volume", "competition_f", "cpc_f", "word_count_f", "has_word_count", "has_volume", "log_age"]
WINDOW = ["log_pre_impr", "log_pre_clk", "pre_ctr_pct", "log_prev30_impr", "prev30_ctr_pct"]
CAT = pd.get_dummies(P[["content_type", "main_intent", "position_tier"]], dtype=int)
X = pd.concat([P[WINDOW + STATIC], CAT], axis=1)
FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label", "health_score", "priority_score", "action_type",
             "ctr", "impressions_90d", "clicks_90d", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]
bad = [c for c in X.columns if c in FORBIDDEN or "last_30d" in c]
assert not bad
print("Feature count:", X.shape[1], "| forbidden columns in features:", bad)
print("Label base rate on the validated population:", round(P["y"].mean(), 3))

Queue population (visible enough before the outcome window): 15,441 pages, 28 clients
  of which 9,102 also have >= 500 impressions in the outcome window (the Week-5/6 validated population)
Feature count: 25 | forbidden columns in features: []
Label base rate on the validated population: 0.361


In [3]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def make_lr(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))
def precision_at_k(scores, labels, k):
    o = np.argsort(-np.asarray(scores), kind="stable")[:k]; return np.asarray(labels)[o].mean()

# Every page is scored by a model that never saw its client (5-fold split by client).
# Training rows: labelled pages of the training clients. Scored rows: ALL queue pages of the test clients.
groups = P["client_id"].values
P["p_under_capture"] = np.nan
fold_rows = []
for i, (tr, te) in enumerate(GroupKFold(5).split(X, groups=groups)):
    assert not set(groups[tr]) & set(groups[te])
    tr_lab = tr[P["labelled"].values[tr]]
    m = make_lr().fit(X.iloc[tr_lab], P["y"].values[tr_lab])
    P.loc[te, "p_under_capture"] = m.predict_proba(X.iloc[te])[:, 1]
    te_lab = te[P["labelled"].values[te]]
    yt = P["y"].values[te_lab]
    fold_rows.append({"fold": i, "test_clients": len(set(groups[te])), "test_labelled": len(te_lab), "base_rate": yt.mean(),
                      "rule_p@50": precision_at_k(P["rule_score"].values[te_lab], yt, 50),
                      "model_p@50": precision_at_k(P["p_under_capture"].values[te_lab], yt, 50)})
folds = pd.DataFrame(fold_rows)
print(folds.round(3).to_string(index=False))
print("\nModel beats the rule at precision@50 in", int((folds['model_p@50'] > folds['rule_p@50']).sum()), "of 5 folds")

# Priority = chance of under-capturing x clicks at stake (both from the feature window)
P["priority_score"] = P["p_under_capture"] * P["est_missed_clicks_30d"]

 fold  test_clients  test_labelled  base_rate  rule_p@50  model_p@50
    0             1           3162      0.464       0.84        0.92
    1             1           2119      0.257       0.72        0.66
    2             7           1502      0.358       0.64        0.80
    3             8            960      0.357       0.70        0.76
    4            11           1359      0.286       0.62        0.82

Model beats the rule at precision@50 in 4 of 5 folds


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### What this queue is

A ranked list of pages a person should look at, with one reason code and one suggested action for each. It is **decision support**: it says which pages look worth a human's time first, and why. It does not say that changing a page will bring clicks back.

**How a page gets its rank.** Every page that had at least 500 impressions in the feature window (days 31-90 back) and a real position is scored: 15,441 pages across 28 clients. A Logistic Regression (the readable model from Week 5) gives each page a chance of under-capturing clicks in the next 30 days, using only feature-window columns. Each page is scored by a model that never saw its client. I then multiply that chance by the **clicks at stake** (clicks the page missed against its tier peers in the feature window, scaled to 30 days). Priority = chance x clicks at stake.

**Why not rank by the model's chance alone?** I tested three rankings on pages with an observed outcome (the Week-5/6 validated population, 9,102 pages; base rate 0.361). Share of the top K that really under-captured, and realized missed clicks per page in the outcome window:

| Ranking | Top 50 | Top 100 | Top 200 | Missed clicks per page, top 50 |
|---|---|---|---|---|
| Rule only (Week 4 logic) | 0.80 | 0.80 | 0.72 | 123.0 |
| Model chance only | 0.90 | 0.87 | 0.86 | 9.5 |
| Chance x clicks at stake (playbook) | 0.90 | 0.90 | 0.88 | 119.8 |

- **Model chance alone** is accurate but picks small pages: right about 9 times in 10, but each page only had about 9.5 missed clicks. That isn't worth an editor's hour.
- **The rule alone** picks the biggest pages (123 missed clicks per page) but is right less often (0.80 at top 50, 0.72 at top 200).
- **The playbook score sits between them:** as accurate as the model at the top and almost as much at stake as the rule. At top 100 it carries about 9% fewer realized missed clicks than the rule alone (8,853 vs 9,709), in exchange for being right about 10 points more often. Neither ranking wins on both counts, and I'm choosing the combination because a review hour is wasted either way on a small page or on a wrong guess.
- **Fold detail:** across the 5 client folds the model beat the rule at precision@50 in 4 of 5. In the fold that was a single client (the second-largest) the rule won, 0.72 vs 0.66.

In [4]:
# ---- Does the ranking carry VALUE, not just accuracy? Evaluate on pages that have an observed outcome. ----
Lv = P[P["labelled"]].copy()
rankings = {"Rule only (W04 logic)": "rule_score", "Model chance only": "p_under_capture", "Chance x clicks at stake (playbook)": "priority_score"}
rows = []
for K in [50, 100, 200]:
    for name, col in rankings.items():
        top = Lv.sort_values(col, ascending=False).head(K)
        rows.append({"K": K, "ranking": name, "share_under_capturing": top["y"].mean(),
                     "mean_realized_missed_clicks_per_page": top["realized_missed_30d"].mean(),
                     "total_realized_missed_clicks": top["realized_missed_30d"].sum()})
val = pd.DataFrame(rows)
print(f"Base rate: {Lv['y'].mean():.3f} | average realized missed clicks per page, all validated pages: {Lv['realized_missed_30d'].mean():.1f}")
print(val.round(2).to_string(index=False))

Base rate: 0.361 | average realized missed clicks per page, all validated pages: 5.2
  K                             ranking  share_under_capturing  mean_realized_missed_clicks_per_page  total_realized_missed_clicks
 50               Rule only (W04 logic)                   0.80                                122.96                       6148.24
 50                   Model chance only                   0.90                                  9.52                        475.86
 50 Chance x clicks at stake (playbook)                   0.90                                119.81                       5990.51
100               Rule only (W04 logic)                   0.80                                 97.09                       9708.93
100                   Model chance only                   0.87                                  9.32                        931.88
100 Chance x clicks at stake (playbook)                   0.90                                 88.53                       8852.7

In [5]:
# ---- ONE primary reason code per page (first rule that applies), then the action it points to ----
# (I first tried a 5th code, "model says risky but no simple rule fires". It matched 1 page out of 15,441, so I dropped it.)
vis_tiers = ["top_3", "page_1", "striking"]
P["fell_over_26pct"] = (P["early_impr"] >= 100) & (P["impressions_prev_30d"] <= 0.74 * P["early_impr"])   # measured inside the feature window

conds = [
    (P["pre_clk"] == 0) & (P["pre_impr"] >= 2000),                                                     # DATA_CHECK_NO_CLICKS
    P["position_tier"].isin(vis_tiers) & (P["pre_ctr_pct"] < 0.5 * P["position_tier"].map(pre_rate) * 100),   # CTR_BELOW_TIER_PEERS
    P["fell_over_26pct"],                                                                               # FADING_IMPRESSIONS
    P["position_tier"].isin(["page_3_5", "deep"]),                                                      # POSITION_LIMITED
]
codes   = ["DATA_CHECK_NO_CLICKS", "CTR_BELOW_TIER_PEERS", "FADING_IMPRESSIONS", "POSITION_LIMITED"]
actions = ["CHECK_TRACKING_FIRST", "REWRITE_TITLE_META", "REVIEW_FOR_REFRESH", "REVIEW_CONTENT_DEPTH"]
P["reason_code"]  = np.select(conds, codes, default="NO_FLAG")
P["action_label"] = np.select(conds, actions, default="MONITOR_ONLY")

print("Pages per reason code (whole queue population):")
print(P["reason_code"].value_counts().to_string())

# Evidence per code, on pages with an observed outcome (out-of-fold model scores; label = under-captured next 30 days)
Lv = P[P["labelled"]]
ev = (Lv.groupby("reason_code")
        .agg(pages=("y", "size"), share_under_capturing=("y", "mean"), mean_model_chance=("p_under_capture", "mean"),
             mean_realized_missed_clicks=("realized_missed_30d", "mean")))
ev["vs_base_rate"] = ev["share_under_capturing"] - Lv["y"].mean()
print("\nEvidence by reason code, outcome = UNDER-CAPTURED CLICKS next 30 days (base rate", round(Lv["y"].mean(), 3), "):")
print(ev.sort_values("share_under_capturing", ascending=False).round(3).to_string())

# FADING is about impressions, not CTR, so it needs its own outcome: did impressions fall >20% next month?
P["impr_fell_20"] = np.where(P["impressions_prev_30d"] >= 500, (P["impressions_last_30d"] < 0.8 * P["impressions_prev_30d"]).astype(float), np.nan)
ev2 = P.groupby("reason_code").agg(pages_with_prev30_ge_500=("impr_fell_20", "count"), share_impressions_fell_20=("impr_fell_20", "mean"))
print("\nSame codes, outcome = IMPRESSIONS FELL >20% next 30 days (all pages with >= 500 impressions in days 31-60; overall", round(P["impr_fell_20"].mean(), 3), "):")
print(ev2.sort_values("share_impressions_fell_20", ascending=False).round(3).to_string())

Pages per reason code (whole queue population):
reason_code
CTR_BELOW_TIER_PEERS    4804
FADING_IMPRESSIONS      4492
NO_FLAG                 3712
POSITION_LIMITED        1981
DATA_CHECK_NO_CLICKS     452

Evidence by reason code, outcome = UNDER-CAPTURED CLICKS next 30 days (base rate 0.361 ):
                      pages  share_under_capturing  mean_model_chance  mean_realized_missed_clicks  vs_base_rate
reason_code                                                                                                     
DATA_CHECK_NO_CLICKS    267                  0.742              0.841                        3.444         0.381
CTR_BELOW_TIER_PEERS   2321                  0.655              0.632                       10.426         0.295
POSITION_LIMITED       1298                  0.382              0.383                        3.529         0.022
FADING_IMPRESSIONS     2164                  0.249              0.258                        1.929        -0.112
NO_FLAG                305

### Reason codes: what each one turned out to mean

Each page gets **one** primary code, the first rule that applies, plus a separate `also_fading` column so the second fact isn't lost. I first tried a fifth code ("model says risky, no simple rule fires"), but it matched 1 page in 15,441, so I dropped it.

| Code | Pages in queue | Under-captured next 30 days (base rate 36.1%) | Impressions fell >20% next 30 days (overall 60.8%) |
|---|---|---|---|
| `DATA_CHECK_NO_CLICKS` | 452 | 74.2% (n=267) | 81.1% (n=424) |
| `CTR_BELOW_TIER_PEERS` | 4,804 | 65.5% (n=2,321) | 66.9% (n=3,099) |
| `POSITION_LIMITED` | 1,981 | 38.2% (n=1,298) | 62.0% (n=1,548) |
| `FADING_IMPRESSIONS` | 4,492 | 24.9% (n=2,164) | 67.0% (n=2,698) |
| `NO_FLAG` | 3,712 | 17.3% (n=3,052) | 47.0% (n=3,350) |

How I read it:

- **`CTR_BELOW_TIER_PEERS` is the code with the strongest evidence for its action.** Pages with a click rate under half of their tier's were far more likely than average to under-capture again (65.5% vs 36.1%).
- **`FADING_IMPRESSIONS` is not a CTR code, so judge it on impressions.** It sits *below* the base rate for under-capturing (24.9%), which is the right reading of a page whose problem is falling visibility. On the outcome it is about, 67.0% of those pages lost more than 20% of impressions, against 47.0% for `NO_FLAG`. Note that `CTR_BELOW_TIER_PEERS` pages lost impressions at almost the same rate (66.9%), so falling impressions is common to both.
- **`POSITION_LIMITED` is close to the base rate for under-capturing** (38.2%). That fits a page on pages 3-5 or deeper whose problem is where it ranks, not its snippet, so a title rewrite isn't the natural first move.
- **`DATA_CHECK_NO_CLICKS`** (zero clicks on 2,000+ impressions): 74.2% kept under-capturing. That fits two stories I can't tell apart from this file: tracking that is broken for those pages, or pages people genuinely don't click. That is exactly why a person checks the data before any action.

In [6]:
# ---- Page profile -> action table (rule-defined profiles; I did NOT cluster, that is a different lane) ----
mapping = pd.DataFrame({
    "reason_code": codes + ["NO_FLAG"],
    "profile": ["Lots of impressions, zero clicks", "Visible position, click rate under half the tier's",
                "Impressions already falling over the feature window", "Ranks on pages 3-5 or deeper",
                "Nothing above applies"],
    "action_label": actions + ["MONITOR_ONLY"],
    "who_decides": ["analyst (check data before anything else)", "content editor", "content editor + analyst",
                    "content editor (not a title problem)", "nobody; re-score next period"]})
evid = ev[["pages", "share_under_capturing"]].rename(columns={"pages": "validated_pages"}).join(ev2["share_impressions_fell_20"])
mapping = mapping.merge(evid, left_on="reason_code", right_index=True, how="left")
mapping["pages_in_queue"] = mapping["reason_code"].map(P["reason_code"].value_counts())
print(mapping[["reason_code", "action_label", "pages_in_queue", "validated_pages", "share_under_capturing", "share_impressions_fell_20", "who_decides"]].round(3).to_string(index=False))
print("\nBy content type (all queue pages), to show what the profiles are made of:")
print(pd.crosstab(P["content_type"], P["reason_code"]).to_string())

         reason_code         action_label  pages_in_queue  validated_pages  share_under_capturing  share_impressions_fell_20                               who_decides
DATA_CHECK_NO_CLICKS CHECK_TRACKING_FIRST             452              267                  0.742                      0.811 analyst (check data before anything else)
CTR_BELOW_TIER_PEERS   REWRITE_TITLE_META            4804             2321                  0.655                      0.669                            content editor
  FADING_IMPRESSIONS   REVIEW_FOR_REFRESH            4492             2164                  0.249                      0.670                  content editor + analyst
    POSITION_LIMITED REVIEW_CONTENT_DEPTH            1981             1298                  0.382                      0.620      content editor (not a title problem)
             NO_FLAG         MONITOR_ONLY            3712             3052                  0.173                      0.470              nobody; re-score next perio

### Page profile to action

I did **not** cluster pages. The skeleton for archetypes belongs to a different lane, so these are rule-defined *profiles*, and the table above shows the mapping:

| Profile (reason code) | Action | Who decides |
|---|---|---|
| Lots of impressions, zero clicks (`DATA_CHECK_NO_CLICKS`) | `CHECK_TRACKING_FIRST` | analyst, before anything else |
| Visible position, click rate under half the tier's (`CTR_BELOW_TIER_PEERS`) | `REWRITE_TITLE_META` | content editor |
| Impressions already falling (`FADING_IMPRESSIONS`) | `REVIEW_FOR_REFRESH` | content editor + analyst |
| Pages 3-5 or deeper (`POSITION_LIMITED`) | `REVIEW_CONTENT_DEPTH` | content editor (not a title problem) |
| Nothing applies (`NO_FLAG`) | `MONITOR_ONLY` | re-score next period |

**I'm not mapping actions by content type.** The queue holds 15,271 keyword articles, 118 feedly articles and only 52 comparison articles (all from one client, as I found in Week 6). That is too thin to say anything by type, so keyword articles are effectively the whole story here.

In [7]:
# ---- Decay / refresh insight, on pages that were visible in the window BEFORE the outcome (outcome-blind selection) ----
D = df[(df["impressions_prev_30d"] >= 500) & (df["avg_position"] > 0)].copy()
D["fell_next_month"] = (D["impressions_last_30d"] < 0.8 * D["impressions_prev_30d"]).astype(int)   # pages that vanished count as fell
print(f"Pages: {len(D):,} | clients: {D['client_id'].nunique()} | share whose impressions fell more than 20% next month: {D['fell_next_month'].mean():.3f}")

# (1) Does STALENESS (days since last update) show up?
order = ["0-30", "31-90", "91-180", "181+"]
stale = D.groupby("freshness_tier").agg(pages=("fell_next_month", "size"), clients=("client_id", "nunique"),
                                        share_fell=("fell_next_month", "mean")).loc[order]
print("\n(1) Fell-next-month rate by days since last update:")
print(stale.round(3).to_string())

# (2) Does MOMENTUM show up? Fade measured only BEFORE the outcome: days 61-90 back  ->  days 31-60 back
D["fade_ratio"] = D["impressions_prev_30d"] / D["early_impr"].where(D["early_impr"] >= 100)
bins = pd.cut(D["fade_ratio"], [-0.001, 0.74, 0.9, 1.1, np.inf], labels=["fell > 26%", "fell 10-26%", "about flat", "rose > 10%"])
D["fade_group"] = bins.astype(object).where(D["fade_ratio"].notna(), "too little history (<100 early impressions)")
mom = D.groupby("fade_group").agg(pages=("fell_next_month", "size"), clients=("client_id", "nunique"), share_fell=("fell_next_month", "mean"))
mom = mom.loc[["fell > 26%", "fell 10-26%", "about flat", "rose > 10%", "too little history (<100 early impressions)"]]
print("\n(2) Fell-next-month rate by the fade BEFORE the outcome window:")
print(mom.round(3).to_string())

# Robustness: is momentum the same inside each client (so it is not one client's story)?
a = D[D["fade_group"] == "fell > 26%"]; b = D[D["fade_group"].isin(["about flat", "rose > 10%"])]
pa = a.groupby("client_id")["fell_next_month"].agg(["size", "mean"]); pb = b.groupby("client_id")["fell_next_month"].agg(["size", "mean"])
both = pa.join(pb, lsuffix="_fading", rsuffix="_flat_or_up", how="inner")
both = both[(both["size_fading"] >= 30) & (both["size_flat_or_up"] >= 30)]
print(f"\nClients with >= 30 pages in BOTH groups: {len(both)} | fading group falls more in {int((both['mean_fading'] > both['mean_flat_or_up']).sum())} of them")
print("Median client-level gap (fading minus flat/up), in percentage points:", round(float((both["mean_fading"] - both["mean_flat_or_up"]).median() * 100), 1))

rng = np.random.default_rng(SEED)
diffs = [a["fell_next_month"].sample(len(a), replace=True, random_state=int(rng.integers(1e9))).mean()
         - b["fell_next_month"].sample(len(b), replace=True, random_state=int(rng.integers(1e9))).mean() for _ in range(500)]
print("Pooled gap (fading minus flat/up): %.3f, bootstrap 95%% range %.3f to %.3f" % (a["fell_next_month"].mean() - b["fell_next_month"].mean(), *np.percentile(diffs, [2.5, 97.5])))

Pages: 11,119 | clients: 28 | share whose impressions fell more than 20% next month: 0.608

(1) Fell-next-month rate by days since last update:
                pages  clients  share_fell
freshness_tier                            
0-30             6513       26       0.606
31-90              35        5       0.543
91-180           4560       15       0.611
181+               11        1       1.000

(2) Fell-next-month rate by the fade BEFORE the outcome window:
                                             pages  clients  share_fell
fade_group                                                             
fell > 26%                                    4321       21       0.701
fell 10-26%                                   1810       21       0.581
about flat                                    1722       23       0.532
rose > 10%                                    3203       26       0.540
too little history (<100 early impressions)     63       14       0.571

Clients with >= 30 pages in 

### Decay and refresh: what I found, including the part that didn't hold

I checked this on 11,119 pages across 28 clients that had at least 500 impressions in days 31-60 back, which is chosen before the outcome so pages that vanished still count. "Fell" means impressions in the last 30 days were under 80% of the 30 days before. Overall, **60.8%** of these pages fell.

**1) Page age since last update: no pattern I can use.** I expected pages not updated for a long time to decay faster.

| Days since last update | Pages | Clients | Share that fell |
|---|---|---|---|
| 0-30 | 6,513 | 26 | 60.6% |
| 31-90 | 35 | 5 | 54.3% |
| 91-180 | 4,560 | 15 | 61.1% |
| 181+ | 11 | 1 | (too few to judge) |

The two big groups (0-30 and 91-180 days) fell at essentially the same rate, 60.6% vs 61.1%. The other two tiers have too few pages to say anything, and the 181+ pages all come from one client. So **this data gives no support for a rule that triggers on page age, and it also can't show that refreshing doesn't work**, because there is almost no group of long-untouched pages to compare against. Also, the dates may reflect how the platform publishes and not an editor's choice, so I can't say what an "update" means here.

**2) Momentum: pages already fading kept fading.** I measured the change in impressions from days 61-90 back to days 31-60 back, which is entirely before the outcome window.

| Change in impressions, days 61-90 to 31-60 back | Pages | Share that fell next month |
|---|---|---|
| Fell more than 26% | 4,321 | 70.1% |
| Fell 10-26% | 1,810 | 58.1% |
| About flat | 1,722 | 53.2% |
| Rose more than 10% | 3,203 | 54.0% |

The gap between fading pages and flat-or-rising pages is **16.4 points** (bootstrap 95% range 14.3 to 18.5). It also holds inside every client I could test: in the 10 clients with at least 30 pages in both groups, fading pages fell more in all 10 (median gap 15.4 points). So this is not one client's story.

What I can say: in this snapshot, pages whose impressions were already falling were **associated with** a higher chance of falling again. What I can't say: that a refresh reverses decay (nobody refreshed pages on purpose in this data, and pages may fade for seasonal reasons I can't see).

**What it means for the playbook:** `REVIEW_FOR_REFRESH` triggers on **fading**, not on age. And it's a "look at this page" flag, not a promise.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

- **Who:** a content analyst and a content editor at a small team, working through a review list on a regular schedule (monthly is what I tested).
- **What for:** deciding which pages deserve a person's time first. The queue is a to-read list, and the action label tells the reader what to look for.
- **What it is not:** an auto-publisher, a forecast of clicks, a statement about what Google rewards, or evidence that any action will work.

### Limits (all of these matter for the paper)

- **One snapshot, one 90-day window, 28 clients.** The biggest client holds about a third of the pages, and 18 clients have at least 30 queue pages. Results can differ on a client the data didn't cover.
- **Validated on 59% of the queue.** The model's accuracy was measured on 9,102 of the 15,441 queue pages, the ones that still had at least 500 impressions in the outcome window. For the other 41% (pages that lost visibility) I have no accuracy number.
- **The label is about click rate against tier peers**, not revenue, leads, or anything a client would call success.
- **Position columns overlap the outcome window** (Week 6). Dropping them changed the result very little, but I still can't remove the overlap from this file.
- **Tier yardsticks come from the same data** they are used on. They are a baseline, not an independent standard.
- **Thin groups:** comparison and feedly articles are too few to say anything about, and the deep tier is a weak label (Week 6).
- **Observed, directional, decision-support.** Nothing here is causal.

In [8]:
# ---- Cost / value thinking: numbers measured on the data, then clearly labelled assumptions ----
K = 50
top = Lv.sort_values("priority_score", ascending=False).head(K)
rand = Lv["realized_missed_30d"].mean()
rule_top = Lv.sort_values("rule_score", ascending=False).head(K)
print(f"MEASURED (pages with an observed outcome), top-{K}:")
print(f"  playbook score : {top['realized_missed_30d'].mean():6.1f} realized missed clicks per page, {top['y'].mean():.0%} truly under-captured")
print(f"  rule only      : {rule_top['realized_missed_30d'].mean():6.1f} realized missed clicks per page, {rule_top['y'].mean():.0%} truly under-captured")
print(f"  average page   : {rand:6.1f} realized missed clicks per page, {Lv['y'].mean():.0%} under-captured")
print("  Both score by clicks at stake, so most of the gap to the average page comes from picking big pages, not from the model.")

# WHAT-IF ARITHMETIC. These are assumptions, NOT findings: nothing in this data shows that a rewrite recovers clicks.
hours_per_page = 1.0
gap = top["realized_missed_30d"].sum()
print("\nWHAT-IF (assumptions, not evidence): suppose a review-and-rewrite takes", hours_per_page, "hour per page.")
for share in [0.0, 0.10, 0.25]:
    print(f"  if a rewrite closed {int(share*100):>2d}% of the gap -> {gap*share:,.0f} extra clicks per 30 days for {K*hours_per_page:.0f} hours "
          f"= {gap*share/(K*hours_per_page):.1f} clicks per hour of work")
print("  Break-even: at 0% recovered, the hours buy nothing. That is a live possibility, and only a test on a small batch can tell.")

# Scope numbers the paper must repeat
print("\nSCOPE")
print(f"  queue pages: {len(P):,} | with a validated outcome: {int(P['labelled'].sum()):,} ({P['labelled'].mean():.0%}) | "
      f"clients in queue: {P['client_id'].nunique()}")
print("  clients with at least 30 queue pages:", int((P['client_id'].value_counts() >= 30).sum()))
print("  position tiers:", P["position_tier"].value_counts().to_dict())

MEASURED (pages with an observed outcome), top-50:
  playbook score :  119.8 realized missed clicks per page, 90% truly under-captured
  rule only      :  123.0 realized missed clicks per page, 80% truly under-captured
  average page   :    5.2 realized missed clicks per page, 36% under-captured
  Both score by clicks at stake, so most of the gap to the average page comes from picking big pages, not from the model.

WHAT-IF (assumptions, not evidence): suppose a review-and-rewrite takes 1.0 hour per page.
  if a rewrite closed  0% of the gap -> 0 extra clicks per 30 days for 50 hours = 0.0 clicks per hour of work
  if a rewrite closed 10% of the gap -> 599 extra clicks per 30 days for 50 hours = 12.0 clicks per hour of work
  if a rewrite closed 25% of the gap -> 1,498 extra clicks per 30 days for 50 hours = 30.0 clicks per hour of work
  Break-even: at 0% recovered, the hours buy nothing. That is a live possibility, and only a test on a small batch can tell.

SCOPE
  queue pages: 15,4

### Cost and value, with measured numbers kept apart from assumptions

**Measured** (pages with an observed outcome, top 50 by the playbook score): on average **119.8** realized missed clicks per page in the next 30 days, against **123.0** for the rule alone and **5.2** for an average validated page. I'm careful with the 5.2 comparison: most of that gap comes from both scores picking big pages, so it is not credit for the model.

**Assumed, not measured.** The what-if lines in the cell above use one hour per page and a recovery share I picked (0%, 10%, 25%). They show how the arithmetic works, not what will happen. Nothing in this file shows that rewriting a title recovers clicks, so 0% is a live possibility.

**Cost of being wrong.** A false alarm costs a reviewer's hour and nothing else, as long as a person reads the page before changing it. A miss means a page with a real click gap isn't looked at, which costs the clicks it could have recovered. Because a false alarm is cheap, I would rather give reviewers a slightly longer list than skip pages.

**The sensible first step:** try the top 20-30 pages from a per-client-capped list, and leave a similar set of flagged pages untouched for comparison. That is the only way to learn whether the actions work. I'm not claiming anything about it until that has been run.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human-review rules

1. **A person reads the page and its search listing before anything changes.** Every row, every action, no exceptions.
2. **`CHECK_TRACKING_FIRST` rows go to the analyst first.** If the zero clicks are a data problem, nothing else on that row means anything.
3. **Review at most 25 pages per client per batch** (use `rank_within_client`). Without it, one client takes over the list (see the cell below).
4. **Read the flags.** `LOW_CLICK_COUNT` (expected under 5 clicks in 30 days): one or two clicks flip the story, as the false alarms in Week 6 did. `THIN_HISTORY` (page under 90 days old, or under 100 impressions in the early window): the earlier record is weak evidence. `DEEP_TIER_LABEL_WEAK`: a deep-tier label effectively means "zero clicks."
5. **Look for a normal reason for the gap:** the answer shows in the search result itself, ads or other features sit above the listing, or the query is branded.
6. **Write down what the person decided and why**, so the monthly re-score can learn from it.

### What should NOT be automated (the no-go list)

- Rewriting or publishing titles and meta descriptions without a person reading the page.
- Deleting, merging, redirecting or unpublishing pages from a score.
- Deciding a page needs a refresh from its age alone (the data shows no age effect).
- Telling a client or a manager to expect a specific click gain.
- Acting on `CHECK_TRACKING_FIRST` pages before the data has been checked.
- Using the queue on a client the model hasn't seen without re-running the Week-6 audit.
- Using it for comparison or feedly articles: there is too little data to say anything about them.
- Making claims about ranking or about what Google rewards.
- Putting client ids, URLs or queries in anything public.

In [9]:
# ---- Human-review flags, computed per page, so the reviewer sees WHY a row needs a closer look ----
P["expected_clicks_30d"] = P["position_tier"].map(bench) * P["pre_impr"] * 30 / FEATURE_DAYS
flags = {
    "LOW_CLICK_COUNT":      P["expected_clicks_30d"] < 5,                  # one or two clicks flip the story (Week-6 false alarms)
    "THIN_HISTORY":         (P["content_age_days"] < 90) | (P["early_impr"] < 100),
    "DEEP_TIER_LABEL_WEAK": P["position_tier"] == "deep",
    "CHECK_DATA_FIRST":     P["reason_code"] == "DATA_CHECK_NO_CLICKS",
}
P["review_flags"] = pd.DataFrame(flags).apply(lambda r: "|".join(k for k, v in r.items() if v) or "STANDARD_REVIEW", axis=1)

Q = P.sort_values("priority_score", ascending=False).reset_index(drop=True)
Q["rank"] = Q.index + 1
Q["rank_within_action"] = Q.groupby("action_label").cumcount() + 1
Q["rank_within_client"] = Q.groupby("client_id").cumcount() + 1       # review rule: at most 25 pages per client per batch
top100 = Q.head(100)
print("Top-100 queue: reason codes")
print(top100["reason_code"].value_counts().to_string())
print("\nTop-100 queue: review flags")
print(top100["review_flags"].value_counts().to_string())
cl = top100["client_id"].value_counts(normalize=True)
print(f"\nTop-100 comes from {top100['client_id'].nunique()} clients; the biggest client holds {cl.iloc[0]:.0%} of it")
capped = Q[Q["rank_within_client"] <= 25].head(100)
cc = capped["client_id"].value_counts(normalize=True)
print(f"With a 25-pages-per-client cap the first 100 come from {capped['client_id'].nunique()} clients (biggest holds {cc.iloc[0]:.0%}); "
      f"{(capped['reason_code'] == 'CTR_BELOW_TIER_PEERS').sum()} are CTR_BELOW_TIER_PEERS")
print("Top-100 pages with no validated outcome (they lost visibility or were not yet scoreable):", int((~top100['labelled']).sum()))

Top-100 queue: reason codes
reason_code
CTR_BELOW_TIER_PEERS    87
POSITION_LIMITED         5
FADING_IMPRESSIONS       4
NO_FLAG                  3
DATA_CHECK_NO_CLICKS     1

Top-100 queue: review flags
review_flags
STANDARD_REVIEW     99
CHECK_DATA_FIRST     1

Top-100 comes from 10 clients; the biggest client holds 54% of it
With a 25-pages-per-client cap the first 100 come from 11 clients (biggest holds 25%); 79 are CTR_BELOW_TIER_PEERS
Top-100 pages with no validated outcome (they lost visibility or were not yet scoreable): 0


### What the top 100 looks like

- 87 of the top 100 are `CTR_BELOW_TIER_PEERS`, 5 are `POSITION_LIMITED`, 4 are `FADING_IMPRESSIONS`, 3 have no code (`NO_FLAG`) and 1 is `DATA_CHECK_NO_CLICKS`. The 3 `NO_FLAG` pages rank high on the model's chance times clicks at stake without any simple rule firing, so a reviewer should treat them as "the model says look, the rules don't."
- **The top 100 comes from 10 clients, and the biggest holds 54% of it.** With the 25-pages-per-client cap, the first 100 come from 11 clients and the biggest holds 25%, with 79 still `CTR_BELOW_TIER_PEERS`. That is why the cap is a review rule, not an option.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers (light, not production)

The table below shows where each number sits in this snapshot and what would make me stop trusting the queue. The thresholds are my judgment calls, not statistics, and I'd revisit them after a few months.

- **Accuracy trigger:** precision@50 on the latest scored month. Over 20 client holdouts, this model's precision@50 averaged 0.79 with a 5th percentile of 0.70, so a month below **0.70** is worse than almost every holdout I saw. You can only measure it after a 30-day wait, because it needs the outcome.
- **Fall back to the rule:** if the model's precision@50 is no better than the plain rule's for two months in a row, go back to the rule. It is simpler and I understand it.
- **Retrain means re-audit.** A retrain is accepted only after the Week-6 checks are re-run (grouped split, leakage audit, failure examples). I would not retrain and ship automatically.
- **Input checks:** the share of the top 100 from one client (54% before the cap today), median earlier CTR, the count of zero-click-high-impression pages (452 today), the share of pages flagged fading (29.1%), and the number of pages visible enough to score (15,441).
- **Calendar:** re-score monthly, re-check the tier yardsticks every quarter.

In [10]:
# ---- Light monitoring: what is normal today, and what would make me stop trusting the queue ----
from sklearn.model_selection import GroupShuffleSplit
Lv = P[P["labelled"]].reset_index(drop=True)
Xl = X[P["labelled"].values].reset_index(drop=True); yl = Lv["y"].values; gl = Lv["client_id"].values
res = []
for tr, te in GroupShuffleSplit(n_splits=20, test_size=0.25, random_state=SEED).split(Xl, yl, gl):
    p = make_lr().fit(Xl.iloc[tr], yl[tr]).predict_proba(Xl.iloc[te])[:, 1]
    res.append((precision_at_k(p, yl[te], 50), yl[te].mean()))
res = np.array(res)
floor = float(np.percentile(res[:, 0], 5))
print(f"Precision@50 over 20 client holdouts: mean {res[:,0].mean():.2f}, 5th percentile {floor:.2f}, min {res[:,0].min():.2f}; base rate mean {res[:,1].mean():.2f}")

today = {
    "precision@50 on the latest scored month":  (round(float(res[:, 0].mean()), 2), f"below {floor:.2f} (the weakest 1-in-20 client holdout)", "RETRAIN + re-audit"),
    "share of top-100 from one client (before the per-client cap)": (round(float(cl.iloc[0]), 2), "rises more than 10 points", "REVIEW CLIENT MIX"),
    "median earlier CTR of queue pages (%)":    (round(float(P["pre_ctr_pct"].median()), 3), "moves by more than 30% either way", "RE-CHECK TIER YARDSTICKS"),
    "pages with 0 clicks and >= 2,000 impressions": (int((P["reason_code"] == "DATA_CHECK_NO_CLICKS").sum()), "doubles", "CHECK TRACKING"),
    "share of queue pages flagged FADING":      (round(float((P["reason_code"] == "FADING_IMPRESSIONS").mean()), 3), "moves by more than 10 points", "RE-TEST MOMENTUM CLAIM"),
    "queue pages (visible enough to score)":    (int(len(P)), "falls by more than 25%", "CHECK DATA FEED"),
}
mon = pd.DataFrame(today, index=["value_in_this_snapshot", "trigger", "what_to_do"]).T
print("\nTRIGGER TABLE")
print(mon.to_string())
print("\nCalendar triggers: re-score monthly; re-run the full Week-6 audit before any retrain is accepted; re-check the tier yardsticks every quarter.")

Precision@50 over 20 client holdouts: mean 0.79, 5th percentile 0.70, min 0.68; base rate mean 0.33

TRIGGER TABLE
                                                             value_in_this_snapshot                                          trigger                what_to_do
precision@50 on the latest scored month                                        0.79  below 0.70 (the weakest 1-in-20 client holdout)        RETRAIN + re-audit
share of top-100 from one client (before the per-client cap)                   0.54                        rises more than 10 points         REVIEW CLIENT MIX
median earlier CTR of queue pages (%)                                         0.158                moves by more than 30% either way  RE-CHECK TIER YARDSTICKS
pages with 0 clicks and >= 2,000 impressions                                    452                                          doubles            CHECK TRACKING
share of queue pages flagged FADING                                           0.291       

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
# ---- Exports for the paper ----
Q["also_fading"] = Q["fell_over_26pct"]
queue_cols = ["rank", "rank_within_action", "rank_within_client", "content_id", "client_id", "content_type", "position_tier", "avg_position",
              "pre_impr", "pre_clk", "pre_ctr_pct", "p_under_capture", "est_missed_clicks_30d", "priority_score",
              "reason_code", "also_fading", "action_label", "review_flags", "labelled"]
out = Q[queue_cols].rename(columns={"pre_impr": "impressions_feature_window", "pre_clk": "clicks_feature_window",
                                    "pre_ctr_pct": "ctr_pct_feature_window", "labelled": "validated_population"})
out.to_csv("work/outputs/action_queue.csv", index=False)
out.head(100).to_csv("work/outputs/action_queue_top100.csv", index=False)
print("Wrote work/outputs/action_queue.csv", out.shape, "and action_queue_top100.csv", out.head(100).shape)

# ---- Figures (no client ids, no URLs) ----
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

# Figure 1: precision of the top-K, rule vs model vs playbook score
f1 = val.pivot(index="K", columns="ranking", values="share_under_capturing")
fig, ax = plt.subplots(figsize=(6.4, 4.2))
w = 0.26
for j, name in enumerate(rankings):
    ax.bar(np.arange(len(f1.index)) + (j - 1) * w, f1[name].values, width=w, label=name)
ax.axhline(Lv["y"].mean(), color="black", ls="--", lw=1)
ax.text(-0.45, Lv["y"].mean() + 0.02, f"base rate {Lv['y'].mean():.2f}", ha="left", color="black", fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
ax.set_xticks(range(len(f1.index))); ax.set_xticklabels([f"top {k}" for k in f1.index])
ax.set_ylim(0, 1); ax.set_ylabel("share that under-captured\nin the next 30 days")
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=1, frameon=False)
ax.set_title(f"Top-ranked pages, by ranking method (clients not seen in training; {len(Lv):,} pages with an outcome)", fontsize=8)
fig.tight_layout(); fig.savefig("work/figures/w07_topk_precision.png", dpi=160); plt.close(fig)

# Figure 2: momentum
m2 = mom.iloc[:4]
fig, ax = plt.subplots(figsize=(6.0, 3.4))
bars = ax.bar(range(len(m2)), m2["share_fell"].values, color="#4c78a8")
for b_, n_, v_ in zip(bars, m2["pages"].values, m2["share_fell"].values):
    ax.text(b_.get_x() + b_.get_width() / 2, 0.08, f"{v_:.0%}\nn={n_:,}", ha="center", fontsize=9, color="white", fontweight="bold")
ax.axhline(D["fell_next_month"].mean(), color="black", ls="--", lw=1)
ax.text(3.45, D["fell_next_month"].mean() + 0.02, f"all pages {D['fell_next_month'].mean():.0%}", ha="right", fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
ax.set_xticks(range(len(m2))); ax.set_xticklabels(m2.index, fontsize=8); ax.set_ylim(0, 1)
ax.set_ylabel("share whose impressions fell >20%\nin the next 30 days"); ax.set_xlabel("change in impressions in the 30 days BEFORE (vs the 30 before that)")
ax.set_title("Pages already fading were more likely to keep fading", fontsize=9)
fig.tight_layout(); fig.savefig("work/figures/w07_fade_momentum.png", dpi=160); plt.close(fig)

# Figure 3: evidence by reason code
e3 = ev.sort_values("share_under_capturing")
fig, ax = plt.subplots(figsize=(6.6, 3.8))
bars = ax.barh(range(len(e3)), e3["share_under_capturing"].values, color="#72b7b2")
for i_, (b_, n_) in enumerate(zip(bars, e3["pages"].values)):
    ax.text(b_.get_width() + 0.015, i_, f"{b_.get_width():.0%}  (n={n_:,})", va="center", fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
ax.axvline(Lv["y"].mean(), color="black", ls="--", lw=1)
ax.text(Lv["y"].mean() + 0.01, len(e3) - 0.45, f"base rate {Lv['y'].mean():.0%}", fontsize=8, va="bottom")
ax.set_yticks(range(len(e3))); ax.set_yticklabels(e3.index, fontsize=8); ax.set_xlim(0, 1.15)
ax.set_xlabel("share that under-captured clicks in the next 30 days"); ax.set_title("What each reason code turned out to mean\n(FADING_IMPRESSIONS is about impressions, not CTR; see notebook)", fontsize=9)
fig.tight_layout(); fig.savefig("work/figures/w07_reason_code_evidence.png", dpi=160); plt.close(fig)
print("Wrote 3 figures to work/figures/")

metrics = {
    "seed": SEED, "sklearn": sklearn.__version__,
    "queue_pages": int(len(P)), "validated_pages": int(P["labelled"].sum()), "base_rate": round(float(Lv["y"].mean()), 4),
    "queue_split": "5-fold by client; model = Logistic Regression; features from the feature window only",
    "precision_at_k": {f"{r.ranking}|top{int(r.K)}": round(float(r.share_under_capturing), 3) for r in val.itertuples()},
    "mean_realized_missed_clicks_top50_playbook": round(float(top["realized_missed_30d"].mean()), 2),
    "mean_realized_missed_clicks_all_validated": round(float(rand), 2),
    "reason_code_counts": {k: int(v) for k, v in P["reason_code"].value_counts().items()},
    "reason_code_share_under_capturing": {k: round(float(v), 3) for k, v in ev["share_under_capturing"].items()},
    "reason_code_share_impressions_fell_20": {k: round(float(v), 3) for k, v in ev2["share_impressions_fell_20"].items()},
    "rule_only_top50_mean_missed_clicks": round(float(rule_top["realized_missed_30d"].mean()), 2),
    "staleness_fell_rate": {k: [int(r.pages), round(float(r.share_fell), 3)] for k, r in stale.iterrows()},
    "fade_fell_rate": {k: [int(r.pages), round(float(r.share_fell), 3)] for k, r in mom.iterrows()},
    "alarm_floor_precision_at_50": round(floor, 3),
    "top100_biggest_client_share": round(float(cl.iloc[0]), 3),
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f: json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w07_playbook_metrics.json")
print("Exports in work/outputs:", sorted(os.listdir("work/outputs")))

Wrote work/outputs/action_queue.csv (15441, 19) and action_queue_top100.csv (100, 19)
Wrote 3 figures to work/figures/
Wrote work/outputs/w07_playbook_metrics.json
Exports in work/outputs: ['action_queue.csv', 'action_queue_top100.csv', 'w07_playbook_metrics.json']


### What was exported, and what the paper builds on

- `work/outputs/action_queue.csv`: the full ranked queue (15,441 rows) with rank, rank within action, rank within client, reason code, `also_fading`, action label, review flags, and a `validated_population` column. **It stays out of git by design; the notebook regenerates it every run.**
- `work/outputs/action_queue_top100.csv`: the first 100 rows.
- `work/outputs/w07_playbook_metrics.json`: the numbers behind this notebook (commit this one).
- `work/figures/w07_topk_precision.png`, `w07_fade_momentum.png`, `w07_reason_code_evidence.png`: figures with n shown and no client ids or URLs (commit these).

The folder name is lowercase on purpose: the notebooks write to `work/outputs/` and `work/figures/`, so the paper next week should read from those exact paths.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.